# E7 · Ajuste fino de la regularización por etiqueta
**Parte de:** E0.

**Hipótesis.** HS, TR y AG tienen dificultades distintas: TR es la más fácil (depende de menciones e insultos explícitos) y AG la más sutil. Un `C` distinto por etiqueta en la regresión logística, o `min_df=1` para no descartar n-gramas raros, podría ajustar mejor cada una.

Se mide cada modelo por separado (LR y Powerset) para ver el efecto sin el stacking.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **C** | Inverso de la regularización: C alto = modelo más flexible (riesgo de sobreajuste), C bajo = más simple |
| **min_df** | Número mínimo de tweets en que debe aparecer un n-grama para entrar al vocabulario |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
filas = []
for C in [1.0, 2.0, 4.0, 8.0, 16.0]:
    cfg = dict(tb.CFG_BASE, C_LR=C)
    for b in ['LR', 'Powerset']:
        r = tb.evaluar(d, {s: tb.oof(d, b, s, cfg) for s in tb.SEEDS})
        filas.append({'modelo': b, 'C': C, **{k: r[k] for k in ['F1 HS', 'F1 TR', 'F1 AG', 'F1 medio', 'F1 medio ±']}})
    tb.log('C =', C)
sens = pd.DataFrame(filas)
sens.round(4)

[     0s] C = 1.0


[     0s] C = 2.0


[     0s] C = 4.0


[     1s] C = 8.0


[     1s] C = 16.0


,modelo,C,F1 HS,F1 TR,F1 AG,F1 medio,F1 medio ±
0,LR,1.0,0.7879,0.8197,0.7886,0.7987,0.0012
1,Powerset,1.0,0.7941,0.8275,0.7879,0.8032,0.0018
2,LR,2.0,0.7902,0.8222,0.7901,0.8008,0.0019
3,Powerset,2.0,0.7936,0.8290,0.7885,0.8037,0.0029
4,LR,4.0,0.7924,0.8246,0.7899,0.8023,0.0020
5,Powerset,4.0,0.7937,0.8284,0.7889,0.8037,0.0021
6,LR,8.0,0.7899,0.8256,0.7909,0.8021,0.0010
7,Powerset,8.0,0.7909,0.8283,0.7877,0.8023,0.0005
8,LR,16.0,0.7923,0.8243,0.7877,0.8014,0.0017
9,Powerset,16.0,0.7901,0.8278,0.7877,0.8018,0.0017


In [3]:
lr = sens[sens.modelo == 'LR']
C_OPT = {l: float(lr.loc[lr[f'F1 {l}'].idxmax(), 'C']) for l in tb.LABELS}
print('Mejor C por etiqueta (LR):', C_OPT)
cfg_c = dict(tb.CFG_BASE, C_POR_ETIQUETA=C_OPT)
cfg_m = dict(tb.CFG_BASE, WORD=dict(tb.CFG_BASE['WORD'], min_df=1), CHAR=dict(tb.CFG_BASE['CHAR'], min_df=1))
R = {'LR · C=4 (E0)': tb.evaluar(d, {s: tb.oof(d, 'LR', s) for s in tb.SEEDS}),
     f'LR · C por etiqueta {C_OPT}': tb.evaluar(d, {s: tb.oof(d, 'LR', s, cfg_c) for s in tb.SEEDS}),
     'LR · min_df=1': tb.evaluar(d, {s: tb.oof(d, 'LR', s, cfg_m) for s in tb.SEEDS})}
tb.tabla(R)

Mejor C por etiqueta (LR): {'HS': 4.0, 'TR': 8.0, 'AG': 8.0}


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
LR · C=4 (E0),0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
"LR · C por etiqueta {'HS': 4.0, 'TR': 8.0, 'AG': 8.0}",0.7924,0.8280,0.7909,0.8038,0.0010,0.7489,0.804 / 0.805 / 0.802
LR · min_df=1,0.7900,0.8217,0.7901,0.8006,0.0032,0.7526,0.804 / 0.801 / 0.796


## Conclusión: **no se adopta**
- El F1 medio de LR y Powerset varía menos de 0,004 entre C=1 y C=16, y menos de 0,002 entre C=2 y C=8. Coincide con la sensibilidad del notebook original (sección 8.8): la configuración está en una zona plana.
- El C "óptimo" por etiqueta se eligió sobre la misma CV con que se evalúa (es optimista) y aun así su ganancia es del orden de la desviación entre semillas.
- `min_df=1` empeora un poco: los n-gramas que aparecen una sola vez son ruido.
- No se genera submission.